In [8]:
import os
import glob
import requests
import chromadb
from chromadb.utils import embedding_functions

In [9]:
CLASS_FILES_DIR = "class_files"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "gemma3:12b"
OLLAMA_BASE_URL = "http://localhost:11434"  # Default Ollama URL
EMBEDDING_DIMENSION = 768  # Dimension of nomic-embed-text embeddings (adjust if needed)
CHROMA_PERSIST_DIR = "chroma_db"
COLLECTION_NAME = "class_files_collection"
N_RESULTS = 10

In [10]:
def read_rst_file(filepath):
    """Reads an RST file and returns its content."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return None

def get_embedding(text):
    """Gets the embedding for the given text using Ollama."""
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    data = {
        "model": EMBEDDING_MODEL,
        "prompt": text
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        result = response.json()
        return result['embedding']
    except requests.exceptions.RequestException as e:
        print(f"Error getting embedding from Ollama: {e}")
        return None

def query_llm(prompt):
    """Queries the LLM using Ollama."""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    data = {
        "model": LLM_MODEL,
        "prompt": prompt,
        "stream": False  # Set to True for streaming output
    }
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()
        result = response.json()
        return result['response']
    except requests.exceptions.RequestException as e:
        print(f"Error querying LLM from Ollama: {e}")
        return None

def chunk_text(text, chunk_size=1000, chunk_overlap=100):
    """Simple text chunking function."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks

def create_chroma_client(persist_directory=CHROMA_PERSIST_DIR):
    """Creates and returns a ChromaDB client."""
    return chromadb.PersistentClient(path=persist_directory)

def get_chroma_collection(client, collection_name=COLLECTION_NAME):
    """Gets or creates a ChromaDB collection."""
    return client.get_or_create_collection(name=collection_name)

In [11]:
# Create ChromaDB client and collection
client = create_chroma_client()
collection = get_chroma_collection(client)

# Index the RST files
if not os.path.exists(CLASS_FILES_DIR):
    print(f"Error: Directory '{CLASS_FILES_DIR}' not found. Please create it and add your RST files.")
else:
    rst_files = glob.glob(os.path.join(CLASS_FILES_DIR, "*.rst"))
    if not rst_files:
        print(f"No RST files found in '{CLASS_FILES_DIR}'.")
    else:
        print("Indexing RST files...")
        for filepath in rst_files:
            filename = os.path.basename(filepath)
            content = read_rst_file(filepath)
            if content:
                chunks = chunk_text(content)
                for i, chunk in enumerate(chunks):
                    embedding = get_embedding(chunk)
                    if embedding:
                        doc_id = f"{filename}_chunk_{i}"
                        collection.add(
                            ids=[doc_id],
                            embeddings=[embedding],
                            documents=[chunk],
                            metadatas={"source": filename, "chunk": i}
                        )
        print("Indexing complete.")

Insert of existing embedding ID: obstacle_avoidance.rst_chunk_0
Add of existing embedding ID: obstacle_avoidance.rst_chunk_0


Indexing RST files...


Insert of existing embedding ID: obstacle_avoidance.rst_chunk_1
Add of existing embedding ID: obstacle_avoidance.rst_chunk_1
Insert of existing embedding ID: obstacle_avoidance.rst_chunk_2
Add of existing embedding ID: obstacle_avoidance.rst_chunk_2
Insert of existing embedding ID: obstacle_avoidance.rst_chunk_3
Add of existing embedding ID: obstacle_avoidance.rst_chunk_3
Insert of existing embedding ID: obstacle_avoidance.rst_chunk_4
Add of existing embedding ID: obstacle_avoidance.rst_chunk_4
Insert of existing embedding ID: custom_functions.rst_chunk_0
Add of existing embedding ID: custom_functions.rst_chunk_0
Insert of existing embedding ID: custom_functions.rst_chunk_1
Add of existing embedding ID: custom_functions.rst_chunk_1
Insert of existing embedding ID: encoders.rst_chunk_0
Add of existing embedding ID: encoders.rst_chunk_0
Insert of existing embedding ID: encoders.rst_chunk_1
Add of existing embedding ID: encoders.rst_chunk_1
Insert of existing embedding ID: encoders.rst_ch

Indexing complete.


In [12]:
query = input("Ask a question about the class files: ")

# Get embedding for the query
query_embedding = get_embedding(query)
retrieved_chunks = None

if query_embedding:
    # Search ChromaDB for relevant documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=N_RESULTS
    )

    if results and results['documents'] and results['documents'][0]:
        retrieved_chunks = results['documents'][0]
        print("\nRetrieved Chunks:")
        for i, chunk in enumerate(retrieved_chunks):
            print(f"--- Chunk {i+1} ---")
            print(chunk)
        context = "\n\n".join(retrieved_chunks)
    else:
        print("No relevant documents found for your query.")
else:
    print("Could not generate embedding for your query.")


Retrieved Chunks:
--- Chunk 1 ---
ack).


Line following – 

On/Off Line Following Concept:
In line following using On/Off control, the XRPLib's reflectance sensor is utilized to detect the line. The sensor provides a value between 0 (black, indicating the line) and 1 (white, off the line). The robot follows the edge of the line, using an if / else statement to determine its course. If the sensor reads a value closer to white, the robot adjusts its course to the left; if closer to black, it adjusts to the right. This method invo
--- Chunk 2 ---
     line is two sensor line following able to handle that one sensor line
      following cannot?

--- Chunk 3 ---
erential in efforts to steer the robot back towards the line. Tuning the proportional gain (KP) is critical, as it determines the robot's responsiveness to deviations from the line. A higher KP results in more aggressive corrections for small deviations, while a lower KP results in gentler corrections."

Example code for line foll

In [13]:
if retrieved_chunks:
    # Formulate the prompt for the LLM
    prompt = f"""You are a helpful assistant. Use the following context from class files to answer the user's question. If you don't know the answer, just say you don't know.

    Context:
    {context}

    Question: {query}"""

    # Query the LLM
    print("\nGenerating answer...")
    answer = query_llm(prompt)
    if answer:
        print("\nAnswer:")
        print(answer)
    else:
        print("Could not get an answer from the LLM.")


Generating answer...

Answer:
The provided text gives example code for line following with one sensor and a proportional controller, but it's incomplete. Here's the code snippet provided, with some explanations:

```python
from XRPLib.defaults import *
from time import sleep

while True:
    error = reflectance.get_left() - 0.5
    #The error variable is the difference between the reading from the left reflectance sensor and 0.5.
    #A value of 0.5 represents the midpoint between black (0) and white (1), and indicates that the robot is centered on the line.
    #The robot adjusts its speed based on the error value to correct its position.
    #A positive error means that the robot is too far to the left, and the robot needs to steer to the right.
    #A negative error means that the robot is too far to the right, and the robot needs to steer to the left.
    #This is a proportional controller.
```

**Important Considerations:**

*   **Missing Code:** The code snippet is incomplete. I